In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import squidpy as sq
import cellcharter as cc
import scanpy as sc
from sklearn.preprocessing import StandardScaler


In [ ]:
## This follows CellCharter, sourced from this paper: https://www.nature.com/articles/s41588-023-01588-4

### Running a Gaussian Mixture Model to predict the clustering number

In [ ]:
### Should just be the columns you wish to cluster (morph + intensities)

numeric_df = comb_columns.select_dtypes(include=[np.number])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(numeric_df)
X_scaled

In [ ]:
### Bic Scores
from sklearn.mixture import GaussianMixture
bics = []
ks = range(1, 11)
scores = []
for k in ks:
    gm = GaussianMixture(n_components=k, random_state=42, covariance_type="diag", n_init=3)
    labels = gm.fit(X_scaled)
    scores.append(labels)
    bics.append(gm.bic(X_scaled))
    print(gm.bic(X_scaled))

best_k = ks[np.argmin(bics)]

gm = GaussianMixture(n_components=best_k, random_state=42).fit(X_scaled)

In [ ]:
## Stability scores (test how stable your clusters are)
from sklearn.metrics import adjusted_rand_score

stability_scores = []
for i in range(len(scores)):
       stability_scores.append(adjusted_rand_score(scores[i], scores[i+1])
                               
mean_score = np.mean(stability_scores)

In [ ]:
from kneed import KneeLocator

### Plot an elbow curve

x = np.arange(1, len(bics) + 1)

kl = KneeLocator(list(x), bics, curve="convex", direction="decreasing")
print(f"Optimal k: {kl.elbow}") 

plt.plot(x, bics, marker='o')
plt.axvline(x = kl.elbow, color = 'r', label = 'axvline - full height', linestyle = 'dashed')
plt.xlabel("Iteration")
plt.xticks(x)
plt.ylabel("Score")
plt.title("Score progression")

plt.savefig(
    "All_metrics_for_pT_pN_Loc/bic_scores.svg",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

### End of the pre-clustering analysis

In [ ]:
from anndata import AnnData
### Create your adata

In [ ]:
random_state=42
sc.pp.scale(adata, max_value=10)
adata.X

In [ ]:
sc.tl.pca(adata, random_state=random_state)
sc.pp.neighbors(adata, random_state=42, n_neighbors = 20)

In [ ]:
sq.gr.spatial_neighbors(adata, library_key='ROI', coord_type='generic', delaunay=False, n_neighs=20)

In [ ]:
library_ids = adata.obs['filename'].unique()

adata.uns['spatial'] = {ids: {} for ids in library_ids}

In [ ]:
sq.pl.spatial_scatter(
    adata_sub,
    color='filename',
    connectivity_key='spatial_connectivities',
    library_id='choose_your_ROI',
    img=False
)

In [ ]:
cc.gr.remove_long_links(adata)

In [ ]:
sq.pl.spatial_scatter(
    adata_sub,
    color='filename',
    connectivity_key='spatial_connectivities',
    library_id='choose_your_ROI',
    img=False
)

In [ ]:
cc.gr.aggregate_neighbors(adata, n_layers=3)

In [ ]:
### Choosing the optimal K
model_params = {
        'random_state': 42,
        'trainer_params': {
            'accelerator':'gpu',
            'enable_progress_bar': True
        },
    }

models = cc.tl.ClusterAutoK(n_clusters=(2,10), model_class=cc.tl.GaussianMixture, model_params=model_params, max_runs=5)

In [ ]:
##Spatial Leiden Clustering
import scanpy as sc

## Cellcharter clustering method
gmm = cc.tl.Cluster(
    n_clusters=3, 
    random_state=12345,
    # If running on GPU
    #trainer_params=dict(accelerator='gpu', devices=1)
)

In [ ]:
### Make sure you have a GPU
from lightning.pytorch import Trainer
Trainer(accelerator="gpu", devices=1)

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
### When using models you find a best K
models.fit(adata, use_rep='X_cellcharter')
adata.obs['spatial_cluster'] = gmm.predict(adata, use_rep='X_cellcharter')

In [ ]:
models.best_k ### Choose 3 for spatial clustering

In [ ]:
gmm.fit(adata, use_rep='X_cellcharter')
adata.obs['spatial_cluster'] = gmm.predict(adata, use_rep='X_cellcharter')

In [ ]:
sq.pl.spatial_scatter(
    adata_sub,
    color='spatial_cluster',
    connectivity_key='spatial_connectivities',
    library_id = 'TMA0_71',
    img=False
)

In [ ]:
sq.pl.spatial_scatter(
    adata_sub,
    color='new_cluster',
    connectivity_key='spatial_connectivities',
    library_id='choose_your_ROI',
    img=False
)

In [ ]:
cc.gr.enrichment(adata, group_key='spatial_cluster', label_key='new_cluster')
cc.pl.enrichment(adata, group_key='spatial_cluster', label_key='new_cluster', figsize=(6,6), fontsize=6, dot_scale=7)

In [ ]:
#pn1 = adata[adata.obs['condition'] == 3]
cc.gr.nhood_enrichment(
    adata,
    cluster_key='spatial_cluster',
)

cc.pl.nhood_enrichment(
    adata,
    cluster_key='spatial_cluster',
    annotate=True,
    vmin=-1,
    vmax=1,
    figsize=(3,3),s
    fontsize=5,
)

plt.savefig("../nhood_enrichment.svg", format="svg", bbox_inches="tight")

In [ ]:
cc.gr.diff_nhood_enrichment(
    adata,
    cluster_key='spatial_cluster',
    condition_key='pN',
    n_perms=150

)

cc.pl.diff_nhood_enrichment(
    adata,
    cluster_key='spatial_cluster',
    condition_key='pN',
    annotate=True,
    vmin=-1,
    vmax=1,
    fontsize=5,
)

plt.savefig("../diff_nhood_enrichment.svg", format="svg", bbox_inches="tight")

In [ ]:
cc.gr.connected_components(adata, cluster_key='spatial_cluster')

In [ ]:
cc.tl.boundaries(adata, min_hole_area_ratio=0.01)

In [ ]:
cc.tl.linearity(adata)
cc.tl.curl(adata)